#训练FP32 CNN model

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import copy

# ==========================================
# 1. 扩容后的 4 层硬件友好 CNN (VGG-Lite)
# ==========================================
class HardwareFriendlyCNN(nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        
        # Layer 1: 3 -> 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False) # 训练时不要bias，由BN代劳
        self.bn1 = nn.BatchNorm2d(64)
        
        # Layer 2: 64 -> 64
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2) # 输出 16x16
        
        # Layer 3: 64 -> 64
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        
        # Layer 4: 64 -> 64
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2) # 输出 8x8
        
        self.fc = nn.Linear(64 * 8 * 8, 10) 
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool1(self.relu(self.bn2(self.conv2(x))))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool2(self.relu(self.bn4(self.conv4(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(x)  
        x = self.fc(x)
        return x

# ==========================================
# 神级工具：离线算子融合 (BN 折叠进 Conv)
# ==========================================
def fuse_conv_bn(conv, bn):
    """
    将 BatchNorm 的参数吸收到 Conv 的 Weight 和 Bias 中，
    使得推理时完全不需要 BN 层！硬件 0 负担！
    """
    # 创建一个新的带有 Bias 的 Conv2d
    fused_conv = nn.Conv2d(
        conv.in_channels, conv.out_channels, 
        kernel_size=conv.kernel_size, stride=conv.stride, 
        padding=conv.padding, bias=True
    )
    
    # 提取 BN 参数
    gamma = bn.weight
    beta = bn.bias
    mean = bn.running_mean
    var = bn.running_var
    eps = bn.eps

    # 提取 Conv 参数
    w_conv = conv.weight.clone()
    b_conv = torch.zeros(conv.out_channels, device=w_conv.device) # 原来没有bias

    # 计算融合乘数
    std = torch.sqrt(var + eps)
    multiplier = gamma / std

    # 融合权重: W_new = W_old * (gamma / std)
    w_fused = w_conv * multiplier.view(-1, 1, 1, 1)
    # 融合偏置: B_new = (B_old - mean) * (gamma / std) + beta
    b_fused = (b_conv - mean) * multiplier + beta

    # 赋值给新的 Conv
    fused_conv.weight.data = w_fused
    fused_conv.bias.data = b_fused

    return fused_conv

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ==========================================
    # 2. 数据增强 (Data Augmentation)
    # ==========================================
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), 
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(), 
    ])

    print("Loading datasets...")
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

    # ==========================================
    # 3. 训练模型
    # ==========================================
    model = HardwareFriendlyCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    epochs = 50 
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        scheduler.step()
        train_acc = 100 * correct_train / total_train
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}, Train Acc: {train_acc:.2f}%, LR: {scheduler.get_last_lr()[0]:.6f}")

    # ==========================================
    # 4. 在测试集上进行全局准确率验证
    # ==========================================
    print("Testing FP32 model accuracy on 10000 test images...")
    model.eval() 
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"==================================================")
    print(f"Original Model Accuracy with BN: {accuracy:.2f} %")
    print(f"==================================================")

    # ==========================================
    # 5. 架构师的魔法：融合 BN，导出干净权重！
    # ==========================================
    print("\nFusing BatchNorm into Convolution layers for Hardware...")
    model.cpu()
    
    # 建立一个只有纯 Conv2d(带bias) 的导出字典
    fused_state_dict = {}
    
    # 融合第一层
    fused_conv1 = fuse_conv_bn(model.conv1, model.bn1)
    fused_state_dict['conv1.weight'] = fused_conv1.weight
    fused_state_dict['conv1.bias'] = fused_conv1.bias
    
    # 融合第二层
    fused_conv2 = fuse_conv_bn(model.conv2, model.bn2)
    fused_state_dict['conv2.weight'] = fused_conv2.weight
    fused_state_dict['conv2.bias'] = fused_conv2.bias
    
    # 融合第三层
    fused_conv3 = fuse_conv_bn(model.conv3, model.bn3)
    fused_state_dict['conv3.weight'] = fused_conv3.weight
    fused_state_dict['conv3.bias'] = fused_conv3.bias
    
    # 融合第四层
    fused_conv4 = fuse_conv_bn(model.conv4, model.bn4)
    fused_state_dict['conv4.weight'] = fused_conv4.weight
    fused_state_dict['conv4.bias'] = fused_conv4.bias
    
    # 拷贝全连接层 (FC 本来就没有 BN，直接拿过来)
    fused_state_dict['fc.weight'] = model.fc.weight
    fused_state_dict['fc.bias'] = model.fc.bias

    # 彻底保存！这才是你的 C 代码提取工具应该去读的权重！
    torch.save(fused_state_dict, 'my_hardware_cnn_fp32_64ch_fused.pth')
    print("Fused weights (Pure Conv+Bias) saved to 'my_hardware_cnn_fp32_64ch_fused.pth'")
    print("Hardware side is completely unaware of BatchNorm! Excellent Co-design!")

if __name__ == '__main__':
    main()

Using device: cuda
Loading datasets...
Files already downloaded and verified
Files already downloaded and verified
Starting training for 50 epochs...
Epoch 1/50, Loss: 1.5543, Train Acc: 43.77%, LR: 0.000999
Epoch 2/50, Loss: 1.1697, Train Acc: 58.17%, LR: 0.000996
Epoch 3/50, Loss: 1.0155, Train Acc: 64.06%, LR: 0.000991
Epoch 4/50, Loss: 0.9210, Train Acc: 67.47%, LR: 0.000984
Epoch 5/50, Loss: 0.8598, Train Acc: 69.88%, LR: 0.000976
Epoch 6/50, Loss: 0.8101, Train Acc: 71.54%, LR: 0.000965
Epoch 7/50, Loss: 0.7693, Train Acc: 73.09%, LR: 0.000952
Epoch 8/50, Loss: 0.7373, Train Acc: 74.07%, LR: 0.000938
Epoch 9/50, Loss: 0.7117, Train Acc: 75.26%, LR: 0.000922
Epoch 10/50, Loss: 0.6878, Train Acc: 75.99%, LR: 0.000905
Epoch 11/50, Loss: 0.6694, Train Acc: 76.83%, LR: 0.000885
Epoch 12/50, Loss: 0.6462, Train Acc: 77.29%, LR: 0.000864
Epoch 13/50, Loss: 0.6261, Train Acc: 78.40%, LR: 0.000842
Epoch 14/50, Loss: 0.6169, Train Acc: 78.64%, LR: 0.000819
Epoch 15/50, Loss: 0.5992, Train 